In [ ]:
from transformers import pipeline
from pathlib import Path
from bs4 import BeautifulSoup
import json
import numpy as np
from pypdf import PdfReader

In [ ]:
#A BERT-model trained by KB to perform Named Entity Recognition in Swedish.
nlp = pipeline('ner', model='KB/bert-base-swedish-cased-ner', tokenizer='KB/bert-base-swedish-cased-ner')

locations_by_year = {}

In [ ]:
class Record:
    """
    Reads a filepath and extracts the locations mentioned in the file content.
    
    Attributes:
    path (str): The filepath.
    year (str): The year of the debate recorded in the file.
    loclist (list): All the locations mentioned in the content of the file.
    """
    def __init__(self, path):
        self.path = path
        self.year = ""
        self.loclist = []

    #In order to identify the year of the record in question, the class extracts the info from the filename.
        pathparts = self.path.split("prot-")
        yearstring = pathparts[1]
        self.year = yearstring[:4]

#In order to identify the locations meantioned in the file, the class employs the BERT-model imported earlier.
        with open(self.path, "r", encoding="utf8") as f:
            data = f.read()

        statements = data.split(" ")

        for statement in statements:
            processed_statement = nlp(statement)
            for named_entity in processed_statement:
                
            #NER returns a list of dictionaries for each identified entity. One of the items of the dictionary identifies the
            # type of entity, designated as one of the values.    
                if "LOC" in named_entity.values():
                    for key, val in named_entity.items():
                    #For many entities, the model splits the word into smaller parts. Here, the subsequent parts are added to the previous word
                    #and are not treated as their own location in the location list. In rare instances, the identified entities begins with a
                    #partial word, in which case the error notification notes the crucial parts of the filename.
                        if key == "word":
                            if "##" in val:
                              try:
                                self.loclist[-1] = self.loclist[-1] + val[2:]
                              except IndexError:
                                print("Problem occured in " + self.path[-16:])
                            else:
                              self.loclist.append(val)
        pass

    
    def store(self):
        """
        Adds the Record information to the comprehensive dictionary.
        """
        if self.year not in locations_by_year.keys():
            locations_by_year[self.year] = self.loclist
        else:
            for key, val in locations_by_year.items():
                if key == self.year:
                    val.extend(self.loclist)


    def __str__(self):
        return self.loclist
        pass

In [ ]:
#When all records has been stored to the locations_by_year dict, it can be stored as a .txt file for later use (manageable on a home PC).
#The following two classes are designed to read this json written .txt file and extract its most relevant information.

class Year:
    """
    Stores the relevant information of a particular year
    
    Attributes:
    year (str): The years when the locations were mentioned.
    locations (list): A list of locations mentioned in the debates.
    frequency (dict): A dict of every location name mentioned, paired with the number of times mentioned.
    """
    def __init__(self, year, locations):
        self.year = year
        self.locations = locations
        self.frequency = {}

        #In order to identify the most common locations, the class pairs each unique location mentioned with its .count().
        for loc in self.locations:
            if loc not in self.frequency.keys():
                self.frequency[loc] = self.locations.count(loc)
            
        pass
    
    def toplist(self, range):
        """
        Displays the top n results of the year together with their frequencies.
        
        Args:
        range (int): Designates how many of the top results that will be presented.
        
        Returns:
        The most mentioned locations printed in sorted order.
        """
        #First the function sorts the frequency dictionary to present the most mentioned locations first.
        entities = list(self.frequency.keys())
        frequencies = list(self.frequency.values())
        sort_index = np.argsort(frequencies)
        descending_frequencies = {entities[i]:frequencies[i] for i in sort_index[::-1]}
        
        range_count = 0
        for key, val in descending_frequencies.items():
            print(f"{key}({val})", end=" ")
            range_count += 1
            #When the count reaches the designated range, the function stops.
            if range_count == range:
                break
            #The linebreaks are designed to fit somewhat neatly in my screen as i open up the result in a new, full-sized window.
            elif range_count % 12 == 0:
                print("\n")
        
        pass



In [ ]:
#The second of the classes designed to read the json .txt files.
class Period:
    """
    Reads a json .txt file and stores its information in a number of Year classes in order to present their most relevant information.
    
    Attributes:
    path (str): The filepath.
    sum (list[Year]): A list of all the years studied in the project.
    """
    def __init__(self, path):
        self.path = path
        self.sum = []

        with open(self.path, 'r') as read_file:
            locations_every_year = json.load(read_file)

        for key, val in locations_every_year.items():
            year_info = Year(key, val)
            self.sum.append(year_info)
                
        pass

    def summary(self, range):
        """
        Presents the toplist of every year in print.
    
        Args:
        range (int): Designates how many results that are returned for each year.
    
        Returns:
        A printed out table of the most references locations of each year and their frequency.
        """
        
        for year in self.sum:
            print(f"""
                      
     - {year.year} - 
                      """)
            year.toplist(range)
        pass
    
    def locations(self, startyear=1887, endyear=1914):
        """
        Identifies the location names referenced at least once in a span of years.
        
        Args:
        startyear (int): The first year in the range, default set to the beginning of the period of my current project.
        endyear (int): The last year in the range, default set to the end of the period of my current project.
        
        Returns:
        A printed display of the location names mentioned in the period.
        """
        location_names = []
        for year in self.sum:
        #The year of each item on the sum list is checked towards the designated range and its locations not yet added to location_names
        #are added.   
            intyear = int(year.year)
            if startyear <= intyear <= endyear:
                for location in year.locations:
                    if location not in location_names:
                        location_names.append(location)
        

        location_names.sort()
        
        #Similarly as for the toplist function, the location names are presented in rows of 12.
        line_counter = 0
        print(f"Number of unique references: {len(location_names)}")
        for reference in location_names:
             print(reference, end=" ")
             line_counter += 1
             if line_counter % 12 == 0:
                  print("\n")
        pass
    
    #I added one function just to easily compare my findings to PDF:s of historical documents outside of parliament debate.
    def pdf(self, path):
        """
        Compares the locations identified with other historical documents.
        
        Args:
        path (str): The path of the PDF file.
        
        Returns:
        A printed display of the parliament location names also mentioned in the historical document.
        """
        #The function creates a list of the location names.
        referenced_locations = []
        comparetext = ""
        print_list = []
        for year in self.sum:
             for location in year.locations:
                  if location not in referenced_locations:
                       referenced_locations.append(location)
        
        #The function reads the PDF file and stores it as a string.
        comparefile = PdfReader(path)
        for page in comparefile.pages:
             comparetext += page.extract_text()
        
        #The matching locations are added to the print list, which is subsequently sorted and printed in rows of 12.
        for loc in referenced_locations:
            if loc in comparetext:
                print_list.append(loc)
        print_list.sort()
        break_counter = 0
        print(f"Number of matching references: {len(print_list)}")
        for match in print_list:
             print(match, end=" ")
             break_counter += 1
             if break_counter % 12 == 0:
                  print("\n")
                 
        pass

In [ ]:
#The parliament data is downloaded as .xml files in separate folders for each year. The following piece of code reads the files
#and stores their information in a .txt document.
directory = Path(r'directory_path')
for dir in directory.iterdir():
  for file in dir.iterdir():
    if file.is_file():
      str_path = str(file)
      filerecord = Record(str_path)
      filerecord.summary()

with open(r'Parliamentlocations.txt', 'w') as data_store:
     data_store.write(json.dumps(locations_by_year))

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


003
009
010
015
019
026
029
032
047
052
055
057


In [ ]:
#The new filepath can now be fed to a Period class and its information be displayed.
test = Period("Parliamentlocations.txt")